# Hands-on — Aula 01: Arquitetura Transformer

Exploração prática dos blocos do Transformer, com modelos **em português** e
**100% locais** (sem chave de API):

| Seção | Tema | Slide |
|---|---|---|
| 1 | Tokenização BPE vs WordPiece | 11–12 |
| 2 | Embeddings e similaridade de cosseno | 13 |
| 3 | Atenção bidirecional (BERT) vs causal (GPT) | 14–15, 22 |
| 4 | Inferência seq2seq e geração autorregressiva | 8, 21–23 |
| 5 | LoRA na prática: a conta dos parâmetros | 20 |

**Modelos:** `pierreguillou/gpt2-small-portuguese` · `neuralmind/bert-base-portuguese-cased` · `Helsinki-NLP/opus-mt-tc-big-en-pt`

> Pré-requisito: venv do curso ativado e `python setup\baixar_modelos.py --aula 01` já executado.
> Execute as células **na ordem** — as seções reutilizam o que já foi carregado.

In [1]:
import numpy as np
import torch
from transformers import (AutoConfig, AutoModel, AutoModelForCausalLM,
                          AutoTokenizer, MarianMTModel, MarianTokenizer)
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # silencia avisos verbosos da biblioteca
torch.manual_seed(42)             # semente fixa: resultados reprodutíveis
print("Setup pronto — GPU disponível:", torch.cuda.is_available())

Setup pronto — GPU disponível: True


## Seção 1 — Tokenização: texto vira unidades discretas *(slides 11–12)*

Dois tokenizadores treinados em português: o do GPT-2 PT (**BPE**) e o do
BERTimbau (**WordPiece**). O que observar:

1. Palavra frequente vira UM token; palavra rara vira VÁRIAS subpalavras.
2. Erro de digitação não quebra o modelo — vira subpalavras conhecidas.
3. Vocabulários diferentes segmentam o MESMO texto de formas diferentes.
4. O `Ġ` é o espaço no BPE; o `Ã©` em `crÃ©dito` é o BPE byte-level mostrando bytes — não é bug.

In [2]:
tok_gpt = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")
tok_bert = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
print("Tokenizadores carregados (rápido — só o vocabulário, não os pesos).")

Tokenizadores carregados (rápido — só o vocabulário, não os pesos).


In [3]:
FRASES = [
    ("frase comum", "O banco aprovou o pedido de crédito do cliente."),
    ("palavra rara / termo técnico", "O exame indicou hipomagnesemia acentuada no paciente."),
    ("erro de digitação", "Segue a recomendasão para o setor financeiro."),
]

for descricao, frase in FRASES:
    print("=" * 70)
    print(f"CASO: {descricao}")
    print(f'FRASE: "{frase}"')
    for nome, tok in (("GPT-2 PT (BPE)", tok_gpt), ("BERTimbau (WordPiece)", tok_bert)):
        pedacos = tok.tokenize(frase)
        ids = tok.encode(frase, add_special_tokens=False)
        print(f"\n  {nome} -> {len(pedacos)} tokens")
        print(f"  tokens: {pedacos}")
        print(f"  ids:    {ids}")

CASO: frase comum
FRASE: "O banco aprovou o pedido de crédito do cliente."

  GPT-2 PT (BPE) -> 10 tokens
  tokens: ['O', 'Ġbanco', 'Ġaprovou', 'Ġo', 'Ġpedido', 'Ġde', 'ĠcrÃ©dito', 'Ġdo', 'Ġcliente', '.']
  ids:    [47, 6060, 11107, 275, 4858, 261, 10335, 298, 9742, 14]

  BERTimbau (WordPiece) -> 10 tokens
  tokens: ['O', 'banco', 'aprovou', 'o', 'pedido', 'de', 'crédito', 'do', 'cliente', '.']
  ids:    [231, 6465, 11473, 146, 4794, 125, 10640, 171, 9379, 119]
CASO: palavra rara / termo técnico
FRASE: "O exame indicou hipomagnesemia acentuada no paciente."

  GPT-2 PT (BPE) -> 11 tokens
  tokens: ['O', 'Ġexame', 'Ġindicou', 'Ġhipo', 'magn', 'es', 'emia', 'Ġacentuada', 'Ġno', 'Ġpaciente', '.']
  ids:    [47, 10786, 17895, 10244, 13802, 272, 3085, 27427, 325, 10098, 14]

  BERTimbau (WordPiece) -> 12 tokens
  tokens: ['O', 'exame', 'indicou', 'hipo', '##magn', '##ese', '##mia', 'acentu', '##ada', 'no', 'paciente', '.']
  ids:    [231, 10423, 17861, 9935, 13634, 2464, 4322, 11534, 251, 

## Seção 2 — Embeddings: tokens em espaço vetorial *(slide 13)*

Abrimos a matriz de embeddings REAL do GPT-2 PT: sua forma é
(tamanho do vocabulário) × (dimensão do vetor). Depois, medimos similaridade
de cosseno entre palavras — pares relacionados devem ficar mais próximos.

In [4]:
modelo_gpt = AutoModel.from_pretrained("pierreguillou/gpt2-small-portuguese")
matriz = modelo_gpt.wte.weight.detach()  # word token embeddings

print(f"MATRIZ DE EMBEDDINGS: {tuple(matriz.shape)}")
print(f"  -> {matriz.shape[0]} tokens no vocabulário, cada um com um vetor de {matriz.shape[1]} dimensões")

ids_exemplo = tok_gpt.encode(" banco", add_special_tokens=False)
print(f"\nO token ' banco' tem id {ids_exemplo[0]}; início do seu vetor:")
print(f"  {matriz[ids_exemplo[0]][:8].numpy().round(3)} ... (+{matriz.shape[1] - 8} dimensões)")

MATRIZ DE EMBEDDINGS: (50257, 768)
  -> 50257 tokens no vocabulário, cada um com um vetor de 768 dimensões

O token ' banco' tem id 6060; início do seu vetor:
  [-0.112 -0.076  0.048  0.058 -0.021  0.068 -0.36   0.042] ... (+760 dimensões)


In [5]:
def vetor_da_palavra(palavra):
    """Embedding médio dos tokens da palavra (com espaço à frente, padrão BPE)."""
    ids = tok_gpt.encode(" " + palavra, add_special_tokens=False)
    return matriz[ids].mean(dim=0), len(ids)

def cosseno(a, b):
    a, b = a.numpy(), b.numpy()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

PALAVRAS = ["rei", "rainha", "cachorro", "gato", "computador"]
vetores = {}
for p in PALAVRAS:
    vetores[p], n = vetor_da_palavra(p)
    print(f"  '{p}' -> {n} token(s) BPE")

print("\nSIMILARIDADE DE COSSENO (1.0 = idênticos):")
pares = [("rei", "rainha"), ("cachorro", "gato"), ("rei", "cachorro"), ("rainha", "computador")]
for a, b in pares:
    print(f"  cos({a:>8s}, {b:<10s}) = {cosseno(vetores[a], vetores[b]):.3f}")

print("\nPares semanticamente próximos têm cosseno maior — a ponte entre")
print("linguagem e álgebra linear. E este é só o vetor de ENTRADA: nas camadas")
print("do Transformer ele será refinado pelo contexto.")

  'rei' -> 1 token(s) BPE
  'rainha' -> 1 token(s) BPE
  'cachorro' -> 1 token(s) BPE
  'gato' -> 1 token(s) BPE
  'computador' -> 1 token(s) BPE

SIMILARIDADE DE COSSENO (1.0 = idênticos):
  cos(     rei, rainha    ) = 0.755
  cos(cachorro, gato      ) = 0.754
  cos(     rei, cachorro  ) = 0.573
  cos(  rainha, computador) = 0.497

Pares semanticamente próximos têm cosseno maior — a ponte entre
linguagem e álgebra linear. E este é só o vetor de ENTRADA: nas camadas
do Transformer ele será refinado pelo contexto.


## Seção 3 — Atenção bidirecional (BERT) vs causal (GPT) *(slides 14–15 e 22)*

**Parte A:** BERTimbau com `output_attentions=True` — para o token **"ele"**,
quais tokens da frase recebem mais atenção? (correferência)

**Parte B:** a máscara causal do GPT — a matriz triangular que impede ver o futuro.

In [6]:
FRASE = "O cliente reclamou porque ele estava sem acesso ao sistema."

bert = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased",
                                 output_attentions=True, attn_implementation="eager")
bert.eval()

entradas = tok_bert(FRASE, return_tensors="pt")
tokens = tok_bert.convert_ids_to_tokens(entradas["input_ids"][0])
with torch.no_grad():
    saida = bert(**entradas)

camada = 7  # camada intermediária — sintaxe/correferência costumam aparecer aqui
atencao = saida.attentions[camada][0].mean(dim=0)  # média das 12 cabeças

pos_ele = tokens.index("ele")
pesos = atencao[pos_ele].clone()
# [CLS] e [SEP] concentram atenção "de descanso" — tiramos para ver as PALAVRAS
for especial in ("[CLS]", "[SEP]"):
    pesos[tokens.index(especial)] = 0.0

print(f'FRASE: "{FRASE}"')
print(f"\nPara o token 'ele' (posição {pos_ele}), a camada {camada + 1} presta mais atenção em:")
top = torch.topk(pesos, k=4)
for peso, pos in zip(top.values, top.indices):
    marcador = ""
    if pos < pos_ele:
        marcador = "  <-- token ANTERIOR (olhou para trás)"
    elif pos > pos_ele:
        marcador = "  <-- token POSTERIOR (olhou para FRENTE!)"
    print(f"  {tokens[pos]:>12s} (posição {int(pos):2d})  peso {float(peso):.3f}{marcador}")

FRASE: "O cliente reclamou porque ele estava sem acesso ao sistema."

Para o token 'ele' (posição 6), a camada 8 presta mais atenção em:
       cliente (posição  2)  peso 0.085  <-- token ANTERIOR (olhou para trás)
             O (posição  1)  peso 0.080  <-- token ANTERIOR (olhou para trás)
           ele (posição  6)  peso 0.054
           sem (posição  8)  peso 0.039  <-- token POSTERIOR (olhou para FRENTE!)


In [7]:
frase_curta = ["O", "cliente", "reclamou", "porque", "ele", "estava"]
n = len(frase_curta)
mascara = torch.tril(torch.ones(n, n, dtype=torch.int))

print("Máscara CAUSAL do GPT — 1 = pode atender, 0 = futuro bloqueado:\n")
print(" " * 12 + "  ".join(f"{t[:7]:>7s}" for t in frase_curta))
for i, linha in enumerate(mascara):
    print(f"{frase_curta[i]:>10s}  " + "  ".join(f"{int(v):>7d}" for v in linha))

print("\nBERT enxerga a frase inteira (COMPREENSÃO); o GPT só enxerga o passado")
print("(obrigatório para GERAR o próximo token) — slide 22.")

Máscara CAUSAL do GPT — 1 = pode atender, 0 = futuro bloqueado:

                  O  cliente  reclamo   porque      ele   estava
         O        1        0        0        0        0        0
   cliente        1        1        0        0        0        0
  reclamou        1        1        1        0        0        0
    porque        1        1        1        1        0        0
       ele        1        1        1        1        1        0
    estava        1        1        1        1        1        1

BERT enxerga a frase inteira (COMPREENSÃO); o GPT só enxerga o passado
(obrigatório para GERAR o próximo token) — slide 22.


## Seção 4 — Inferência: seq2seq e geração autorregressiva *(slides 8, 21–23)*

**Parte A:** tradução EN→PT com MarianMT (encoder-decoder do Transformer
original) — mostrando `input_ids`, `attention_mask` e a saída decodificada.
O token `>>por<<` indica o idioma-alvo.

**Parte B:** geração autorregressiva com o GPT-2 PT (decoder-only), UM token
por vez, para ver o loop acontecendo.

In [8]:
nome_marian = "Helsinki-NLP/opus-mt-tc-big-en-pt"
tok_m = MarianTokenizer.from_pretrained(nome_marian)
marian = MarianMTModel.from_pretrained(nome_marian)

FRASES_EN = [
    ">>por<< Attention is all you need.",
    ">>por<< The transformer architecture changed natural language processing forever.",
]

for frase in FRASES_EN:
    print("-" * 70)
    print(f'ENTRADA (com token de idioma-alvo): "{frase}"')
    lote = tok_m(frase, return_tensors="pt")
    print(f"  input_ids:      {lote['input_ids'][0].tolist()}")
    print(f"  attention_mask: {lote['attention_mask'][0].tolist()}")
    print(f"  tokens:         {tok_m.convert_ids_to_tokens(lote['input_ids'][0])}")
    with torch.no_grad():
        saida = marian.generate(**lote, max_new_tokens=40)
    print(f"  ids gerados pelo decoder: {saida[0].tolist()}")
    print(f'  TRADUÇÃO: "{tok_m.decode(saida[0], skip_special_tokens=True)}"')

----------------------------------------------------------------------
ENTRADA (com token de idioma-alvo): ">>por<< Attention is all you need."
  input_ids:      [39141, 5950, 28398, 3588, 54605, 34450, 28, 44670]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]
  tokens:         ['>>por<<', '▁Attention', '▁is', '▁all', '▁you', '▁need', '.', '</s>']


  ids gerados pelo decoder: [54775, 5737, 17364, 51422, 35317, 41409, 53508, 39597, 28, 44670]
  TRADUÇÃO: "Atenção é tudo o que você precisa."
----------------------------------------------------------------------
ENTRADA (com token de idioma-alvo): ">>por<< The transformer architecture changed natural language processing forever."
  input_ids:      [39141, 49850, 50889, 5035, 9854, 34291, 29697, 40289, 22398, 28, 44670]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  tokens:         ['>>por<<', '▁The', '▁transformer', '▁architecture', '▁changed', '▁natural', '▁language', '▁processing', '▁forever', '.', '</s>']


  ids gerados pelo decoder: [54775, 1757, 5218, 16785, 50875, 33874, 35317, 40280, 14224, 30582, 34291, 37032, 45688, 28, 44670]
  TRADUÇÃO: "A arquitetura do transformador mudou o processamento da linguagem natural para sempre."


In [9]:
gpt = AutoModelForCausalLM.from_pretrained("pierreguillou/gpt2-small-portuguese")

PROMPT = "A inteligência artificial vai transformar o mercado de trabalho porque"
print(f'PROMPT: "{PROMPT}"')
lote = tok_gpt(PROMPT, return_tensors="pt")

ids = lote["input_ids"]
print("\nGerando um token por vez (cada linha = um passo do loop):")
with torch.no_grad():
    for passo in range(15):
        logits = gpt(ids).logits[0, -1]         # distribuição sobre o vocabulário
        proximo = int(torch.argmax(logits))      # greedy: o token mais provável
        ids = torch.cat([ids, torch.tensor([[proximo]])], dim=1)
        print(f"  passo {passo + 1:2d}: id {proximo:6d} -> '{tok_gpt.decode([proximo])}'")

print(f'\nTEXTO FINAL: "{tok_gpt.decode(ids[0], skip_special_tokens=True)}"')
print("O decoder só usa o que já foi gerado — atenção causal em ação.")

PROMPT: "A inteligência artificial vai transformar o mercado de trabalho porque"

Gerando um token por vez (cada linha = um passo do loop):
  passo  1: id    275 -> ' o'
  passo  2: id   1316 -> ' trabalho'
  passo  3: id    372 -> ' é'
  passo  4: id    447 -> ' mais'
  passo  5: id  26520 -> ' barato'
  passo  6: id    258 -> ' e'
  passo  7: id    447 -> ' mais'
  passo  8: id   7050 -> ' fácil'


  passo  9: id    261 -> ' de'
  passo 10: id    303 -> ' se'
  passo 11: id   1569 -> ' fazer'
  passo 12: id     14 -> '.'
  passo 13: id    199 -> '
'
  passo 14: id    199 -> '
'
  passo 15: id     47 -> 'O'

TEXTO FINAL: "A inteligência artificial vai transformar o mercado de trabalho porque o trabalho é mais barato e mais fácil de se fazer.

O"
O decoder só usa o que já foi gerado — atenção causal em ação.


## Seção 5 — LoRA na prática: a conta dos parâmetros *(slide 20)*

Não treinamos nada — fazemos a CONTA. LoRA congela o modelo e injeta, em cada
matriz adaptada, duas matrizes pequenas A (d×r) e B (r×d). Com r pequeno, a
correção treinável vira uma fração mínima do modelo.

In [10]:
config = AutoConfig.from_pretrained("pierreguillou/gpt2-small-portuguese")
n_camadas, d, vocab = config.n_layer, config.n_embd, config.vocab_size

por_bloco = (
    3 * d * d + 3 * d               # projeções Q,K,V (c_attn)
    + d * d + d                     # projeção de saída da atenção
    + 2 * (d * 4 * d) + 4 * d + d   # FFN (768->3072->768)
    + 4 * d                         # 2 layer norms
)
total = vocab * d + config.n_positions * d + n_camadas * por_bloco + 2 * d

print(f"MODELO: GPT-2 small PT — {n_camadas} camadas, dimensão {d}, vocabulário {vocab}")
print(f"Parâmetros totais (aproximação pela arquitetura): {total:,}".replace(",", "."))
print("\nLoRA aplicada às projeções Q e V de cada camada (receita clássica):")
for r in (4, 8, 16):
    lora = n_camadas * 2 * (d * r + r * d)   # A (d x r) + B (r x d), em Q e V
    print(f"  posto r={r:2d}: {lora:>9,} parâmetros treináveis "
          f"({100 * lora / total:.2f}% do modelo)".replace(",", "."))

print("\nCom r=8, treinamos ~0,2% dos parâmetros e PRESERVAMOS o conhecimento")
print("pré-treinado. Em produção: menos VRAM, checkpoints minúsculos,")
print("vários adapters por modelo.")

MODELO: GPT-2 small PT — 12 camadas, dimensão 768, vocabulário 50257
Parâmetros totais (aproximação pela arquitetura): 124.439.808

LoRA aplicada às projeções Q e V de cada camada (receita clássica):
  posto r= 4:   147.456 parâmetros treináveis (0.12% do modelo)
  posto r= 8:   294.912 parâmetros treináveis (0.24% do modelo)
  posto r=16:   589.824 parâmetros treináveis (0.47% do modelo)

Com r=8, treinamos ~0,2% dos parâmetros e PRESERVAMOS o conhecimento
pré-treinado. Em produção: menos VRAM, checkpoints minúsculos,
vários adapters por modelo.
